# Flow Matching: Root Finding

Find values of $x$ where $f(x) = 0$ for the polynomial $y = x^4 - 4x^2 + x + 3$.

Flow Matching learns a vector field that transforms a simple base distribution (e.g., Gaussian) to the target data distribution. For conditional generation, we condition the flow on desired output values.

**Authors:** Victor Alves and John R. Kitchin

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU for JAX (must be set before importing JAX)
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch

# Import reusable utilities from local module
from generative_optimization import (
    generate_samples,
    cluster_stats,
    ConditionalFlowMatching
)

# Force CPU for PyTorch
device = torch.device('cpu')
print(f"Using device: {device}")

# Figure settings
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Problem Setup

We want to find roots of $f(x) = x^4 - 4x^2 + x + 3 = 0$.

The approach:
1. Generate training data: pairs of $(x, f(x))$
2. Train flow matching to generate $x$ conditioned on $f(x)$
3. Sample by conditioning on $f(x) = 0$

In [ ]:
# Generate data - use more points and oversample near roots
x_base = np.linspace(-2.25, 2, 1000)
y_base = x_base**4 - 4*x_base**2 + x_base + 3

# True roots
roots = np.roots([1, 0, -4, 1, 3])
real_roots = roots[roots.imag == 0].real
print(f"Real roots: {real_roots}")

# Oversample near the roots (where |y| is small)
# Add extra points in regions where y is close to 0
x_near_roots = []
for r in real_roots:
    x_near_roots.append(np.linspace(r - 0.3, r + 0.3, 200))
x_near_roots = np.concatenate(x_near_roots)
y_near_roots = x_near_roots**4 - 4*x_near_roots**2 + x_near_roots + 3

# Combine base and oversampled data
x = np.concatenate([x_base, x_near_roots])
y = np.concatenate([y_base, y_near_roots])

print(f"Total data points: {len(x)}")
print(f"Data points near roots: {len(x_near_roots)}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(x_base, y_base, 'b-', lw=2, label='f(x)')
plt.plot(real_roots, [0, 0], 'ro', ms=10, label='Real roots')
plt.axhline(0, color='k', ls='--', alpha=0.5)
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.title('$y = x^4 - 4x^2 + x + 3$')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Prepare data for flow matching
# x_data: what we want to generate (the input x)
# c_data: what we condition on (the output y)

x_data = x.reshape(-1, 1)
c_data = y.reshape(-1, 1)

print(f"x_data shape: {x_data.shape}")
print(f"c_data shape: {c_data.shape}")

In [ ]:
# Train flow matching model with larger architecture and more epochs
fm_root = ConditionalFlowMatching(x_dim=1, c_dim=1, hidden_dim=128, n_layers=4, sigma_min=0.001)
losses = fm_root.fit(x_data, c_data, epochs=1000, batch_size=128, lr=1e-3)

In [ ]:
# Plot learning curves (loss, MAE, R²)
fm_root.plot_learning_curves()

In [ ]:
# Sample roots by conditioning on y=0
# Use more integration steps for better quality
samples = fm_root.sample(c_values=[[0.0]], n_samples=1000, n_steps=100)

plt.figure(figsize=(8, 4))
plt.hist(samples[:, 0], bins=50, density=True, alpha=0.7)
for r in real_roots:
    plt.axvline(r, color='r', ls='--', lw=2, label=f'True root: {r:.3f}')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Samples conditioned on y=0')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nCluster statistics:")
cluster_stats(samples, eps=0.3)

In [ ]:
# Verify the roots
f_at_samples = samples[:, 0]**4 - 4*samples[:, 0]**2 + samples[:, 0] + 3
print(f"Mean |f(x)| at samples: {np.mean(np.abs(f_at_samples)):.6f}")
print(f"Max |f(x)| at samples: {np.max(np.abs(f_at_samples)):.6f}")

In [ ]:
# Can also condition on other values
samples_y5 = fm_root.sample(c_values=[[5.0]], n_samples=500)

print("Values of x where y=5:")
cluster_stats(samples_y5)

# Verify
f_at_samples = samples_y5[:, 0]**4 - 4*samples_y5[:, 0]**2 + samples_y5[:, 0] + 3
print(f"\nMean y at samples: {np.mean(f_at_samples):.4f} (expected: 5.0)")